# Expense Forecasting Notebook

This notebook focuses on next-month expense forecasting by category using methods that are easy to explain in a finance portfolio:
- **Moving Average**
- **Linear Regression**
- **Optional ARIMA** using `statsmodels`

## Forecasting Goals
- Estimate next month's spend per category
- Compare simple vs trend-aware models
- Evaluate model quality using **MAE** and **RMSE**
- Visualize actual vs predicted behavior


In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (11, 5)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.titleweight"] = "bold"

try:
    from statsmodels.tsa.arima.model import ARIMA
    ARIMA_AVAILABLE = True
except Exception:
    ARIMA_AVAILABLE = False

data_dir = Path("../data")
transactions = pd.read_csv(data_dir / "sample_transactions.csv", parse_dates=["transaction_date"])
expenses = transactions[transactions["type"] == "expense"].copy()
expenses["month"] = expenses["transaction_date"].dt.to_period("M").astype(str)
expenses["month_dt"] = pd.to_datetime(expenses["month"])

monthly_category = (
    expenses.groupby(["month", "category"], as_index=False)["amount"]
    .sum()
    .sort_values(["category", "month"])
)
monthly_category["month_num"] = monthly_category.groupby("category").cumcount()

print("ARIMA available:", ARIMA_AVAILABLE)
monthly_category.head()


## 1. Data Preparation

**Why it matters:** Forecasts become more stable at the monthly-category level because aggregation removes daily noise while preserving spending trends.


In [ ]:
category_summary = (
    monthly_category.groupby("category")
    .agg(months=("month", "nunique"), avg_monthly_spend=("amount", "mean"))
    .reset_index()
    .sort_values("avg_monthly_spend", ascending=False)
)
category_summary["avg_monthly_spend"] = category_summary["avg_monthly_spend"].round(2)
category_summary


**Business insight:** Categories with longer and more stable histories are better candidates for model-based forecasting than sparse or irregular spending categories.


## 2. Moving Average Forecast

**Why it matters:** Moving averages are intuitive and easy to explain to non-technical stakeholders. They work well when spending is relatively stable and recent history is the best predictor.


In [ ]:
def moving_average_forecast(values, window=3):
    values = list(values)
    if len(values) == 0:
        return np.nan
    if len(values) < window:
        return float(np.mean(values))
    return float(np.mean(values[-window:]))

sma_results = []
for category in monthly_category["category"].unique():
    cat = monthly_category[monthly_category["category"] == category].sort_values("month")
    values = cat["amount"].tolist()

    next_forecast = moving_average_forecast(values, window=3)
    actual = np.array(values[3:])
    rolling_preds = np.array([moving_average_forecast(values[:i], window=3) for i in range(3, len(values))])

    mae = mean_absolute_error(actual, rolling_preds) if len(actual) else np.nan
    rmse = np.sqrt(mean_squared_error(actual, rolling_preds)) if len(actual) else np.nan

    sma_results.append({
        "category": category,
        "last_month_actual": values[-1],
        "ma_3_forecast": round(next_forecast, 2),
        "mae_ma": round(mae, 2) if pd.notna(mae) else np.nan,
        "rmse_ma": round(rmse, 2) if pd.notna(rmse) else np.nan,
    })

sma_df = pd.DataFrame(sma_results).sort_values("ma_3_forecast", ascending=False)
sma_df


**Business insight:** Moving average forecasts are especially useful for categories like groceries or utilities where recent behavior tends to repeat without strong trend acceleration.


## 3. Linear Regression Forecast

**Why it matters:** Linear regression adds trend sensitivity. It works better when a category is steadily increasing or decreasing over time.


In [ ]:
lr_results = []
actual_vs_pred_records = []

for category in monthly_category["category"].unique():
    cat = monthly_category[monthly_category["category"] == category].sort_values("month").copy()
    X = cat["month_num"].values.reshape(-1, 1)
    y = cat["amount"].values

    model = LinearRegression()
    model.fit(X, y)
    y_pred = model.predict(X)
    next_forecast = model.predict([[len(cat)]])[0]

    for month, actual, predicted in zip(cat["month"], y, y_pred):
        actual_vs_pred_records.append({
            "category": category,
            "month": month,
            "actual": actual,
            "predicted": predicted,
            "model": "Linear Regression",
        })

    lr_results.append({
        "category": category,
        "last_month_actual": y[-1],
        "lr_forecast": round(float(next_forecast), 2),
        "trend_slope": round(float(model.coef_[0]), 2),
        "mae_lr": round(mean_absolute_error(y, y_pred), 2),
        "rmse_lr": round(np.sqrt(mean_squared_error(y, y_pred)), 2),
    })

lr_df = pd.DataFrame(lr_results).sort_values("lr_forecast", ascending=False)
lr_df


**Business insight:** Trend-aware forecasting is useful for categories like dining, entertainment, or shopping where spending can drift upward over time.


## 4. Optional ARIMA Forecast

**Why it matters:** ARIMA can capture autocorrelation and momentum, giving a more advanced time-series angle for the portfolio. It remains optional because it depends on `statsmodels`.


In [ ]:
if ARIMA_AVAILABLE:
    arima_results = []
    for category in monthly_category["category"].unique():
        cat = monthly_category[monthly_category["category"] == category].sort_values("month")
        y = cat["amount"].values

        try:
            model = ARIMA(y, order=(1, 1, 1))
            fitted = model.fit()
            fitted_values = fitted.predict(start=1, end=len(y) - 1, typ="levels")
            actual_values = y[1:]
            next_forecast = fitted.forecast(steps=1)[0]

            arima_results.append({
                "category": category,
                "arima_forecast": round(float(next_forecast), 2),
                "mae_arima": round(mean_absolute_error(actual_values, fitted_values), 2),
                "rmse_arima": round(np.sqrt(mean_squared_error(actual_values, fitted_values)), 2),
            })
        except Exception:
            arima_results.append({
                "category": category,
                "arima_forecast": np.nan,
                "mae_arima": np.nan,
                "rmse_arima": np.nan,
            })

    arima_df = pd.DataFrame(arima_results)
    arima_df
else:
    print("ARIMA skipped. Install statsmodels to enable this section.")


**Business insight:** ARIMA is helpful when recent spend depends strongly on the previous months rather than following a simple straight-line trend.


## 5. Model Comparison

**Why it matters:** Comparing models side by side helps decide whether a category is best forecast with a stable-history method or a trend-aware method.


In [ ]:
comparison = sma_df.merge(
    lr_df[["category", "lr_forecast", "trend_slope", "mae_lr", "rmse_lr"]],
    on="category",
    how="left",
)

if ARIMA_AVAILABLE:
    comparison = comparison.merge(arima_df, on="category", how="left")

comparison = comparison.sort_values("last_month_actual", ascending=False)
comparison


**Business insight:** Categories with lower MAE and RMSE are more predictable. Those with high error metrics may need richer features, more history, or a different planning approach.


## 6. Actual vs Predicted Visualization

**Why it matters:** Visual comparison makes model behavior easier to interpret for reviewers and hiring managers, especially in a portfolio context.


In [ ]:
viz_category = "Dining" if "Dining" in monthly_category["category"].unique() else monthly_category["category"].iloc[0]
viz_df = pd.DataFrame(actual_vs_pred_records)
viz_df = viz_df[viz_df["category"] == viz_category].copy()
viz_df["month_dt"] = pd.to_datetime(viz_df["month"])

fig, ax = plt.subplots()
sns.lineplot(data=viz_df, x="month_dt", y="actual", marker="o", linewidth=2.5, label="Actual", ax=ax)
sns.lineplot(data=viz_df, x="month_dt", y="predicted", marker="o", linewidth=2.5, label="Predicted", ax=ax)
ax.set_title(f"Actual vs Predicted Spend — {viz_category}")
ax.set_xlabel("Month")
ax.set_ylabel("Amount")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 7. Next-Month Forecast by Category

**Why it matters:** This is the practical output the app can surface in analytics views, budget planning, and proactive overspend warnings.


In [ ]:
forecast_view = comparison.copy()
forecast_columns = ["category", "last_month_actual", "ma_3_forecast", "lr_forecast"]
if ARIMA_AVAILABLE:
    forecast_columns.append("arima_forecast")
forecast_view[forecast_columns]


## Key Takeaways

- Moving average gives a strong simple baseline for stable categories.
- Linear regression is better when category spend shows directional trend.
- ARIMA provides an advanced option when `statsmodels` is available.
- MAE and RMSE make the forecasting work more credible for a data-analyst portfolio because they show the models are evaluated, not just plotted.
